# 📊 Data Exploration, Visualization & Preprocessing
**Datasets:** MNIST · CIFAR-10 · RCV1 · Covtype

**Pipeline:**
1. Load raw data
2. Explore & thống kê mô tả
3. Trực quan hóa
4. Phát hiện vấn đề (missing, imbalance, outlier)
5. Tiền xử lý & Chuẩn hóa
6. Kiểm tra sau chuẩn hóa
7. Lưu data đã xử lý

## 0. Setup & Import

In [ ]:
import os
import gzip
import pickle
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, MaxAbsScaler, Normalizer
)
from sklearn.decomposition import PCA
from collections import Counter

# ── Style ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# ── Base path ──────────────────────────────────────────────────
BASE = r"d:\Khóa luận tốt nghiệp\experiment\data"

print("✅ Imports OK")

---
## 1. Load Functions

In [ ]:
# ── MNIST ──────────────────────────────────────────────────────
def _read_idx_images(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rb') as f:
        magic = int.from_bytes(f.read(4), 'big')
        assert magic == 2051
        n    = int.from_bytes(f.read(4), 'big')
        rows = int.from_bytes(f.read(4), 'big')
        cols = int.from_bytes(f.read(4), 'big')
        data = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows * cols)
    return data.astype(np.float64), rows, cols   # raw pixel (0-255)

def _read_idx_labels(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rb') as f:
        magic = int.from_bytes(f.read(4), 'big')
        assert magic == 2049
        n = int.from_bytes(f.read(4), 'big')
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels.astype(np.int64)

def load_mnist_raw():
    base = os.path.join(BASE, 'mnist')
    X_tr, H, W = _read_idx_images(os.path.join(base,'train-images-idx3-ubyte','train-images.idx3-ubyte'))
    y_tr = _read_idx_labels(os.path.join(base,'train-labels-idx1-ubyte','train-labels.idx1-ubyte'))
    X_te, _, _ = _read_idx_images(os.path.join(base,'t10k-images-idx3-ubyte','t10k-images.idx3-ubyte'))
    y_te = _read_idx_labels(os.path.join(base,'t10k-labels-idx1-ubyte','t10k-labels.idx1-ubyte'))
    return X_tr, y_tr, X_te, y_te, H, W

# ── CIFAR-10 ───────────────────────────────────────────────────
def load_cifar10_raw():
    base = os.path.join(BASE, 'cifar-10-python', 'cifar-10-batches-py')
    Xs, ys = [], []
    for i in range(1, 6):
        with open(os.path.join(base, f'data_batch_{i}'), 'rb') as f:
            d = pickle.load(f, encoding='bytes')
        Xs.append(d[b'data'].astype(np.float64))
        ys.extend(d[b'labels'])
    X_tr = np.vstack(Xs);  y_tr = np.array(ys, dtype=np.int64)
    with open(os.path.join(base, 'test_batch'), 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    X_te = d[b'data'].astype(np.float64)
    y_te = np.array(d[b'labels'], dtype=np.int64)
    labels = ['airplane','automobile','bird','cat','deer',
              'dog','frog','horse','ship','truck']
    return X_tr, y_tr, X_te, y_te, labels

# ── RCV1 ───────────────────────────────────────────────────────
def load_rcv1_raw():
    path = os.path.join(BASE, 'rcv1','rcv1_train.binary','rcv1_train.binary')
    X, y = load_svmlight_file(path)
    y = y.astype(np.float64)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_tr, y_tr, X_te, y_te

# ── Covtype ────────────────────────────────────────────────────
def load_covtype_raw():
    path = os.path.join(BASE, 'covtype','covtype.libsvm.binary','covtype.libsvm.binary')
    X, y = load_svmlight_file(path)
    y_bin = np.where(y == 2, 1.0, -1.0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y_bin, test_size=0.5, random_state=42)
    return X_tr, y_tr, X_te, y_te

print("✅ Loaders defined")

---
## 2. MNIST — Explore & Visualize

In [ ]:
X_tr_mn, y_tr_mn, X_te_mn, y_te_mn, H_mn, W_mn = load_mnist_raw()

print("=" * 55)
print("MNIST — Thống kê cơ bản")
print("=" * 55)
print(f"X_train : {X_tr_mn.shape}  dtype={X_tr_mn.dtype}")
print(f"y_train : {y_tr_mn.shape}  classes={np.unique(y_tr_mn)}")
print(f"X_test  : {X_te_mn.shape}")
print(f"y_test  : {y_te_mn.shape}")
print(f"Pixel range  : [{X_tr_mn.min():.0f}, {X_tr_mn.max():.0f}]")
print(f"Mean pixel   : {X_tr_mn.mean():.4f}")
print(f"Std  pixel   : {X_tr_mn.std():.4f}")
print(f"Missing values: {np.isnan(X_tr_mn).sum()}")

In [ ]:
# ── Phân phối nhãn ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (X, y, split) in zip(axes, [
        (X_tr_mn, y_tr_mn, 'Train'),
        (X_te_mn, y_te_mn, 'Test')]):
    counts = Counter(y)
    ax.bar(counts.keys(), counts.values(), color=sns.color_palette('tab10'))
    ax.set_title(f'MNIST — Phân phối nhãn ({split})')
    ax.set_xlabel('Chữ số'); ax.set_ylabel('Số mẫu')
    ax.set_xticks(range(10))

plt.tight_layout(); plt.show()

In [ ]:
# ── Hiển thị mẫu ảnh ───────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    idx = np.where(y_tr_mn == digit)[0][0]
    img = X_tr_mn[idx].reshape(H_mn, W_mn)
    axes[0, digit].imshow(img, cmap='gray')
    axes[0, digit].set_title(str(digit), fontsize=12)
    axes[0, digit].axis('off')
    # ảnh thứ 2
    idx2 = np.where(y_tr_mn == digit)[0][5]
    axes[1, digit].imshow(X_tr_mn[idx2].reshape(H_mn, W_mn), cmap='gray')
    axes[1, digit].axis('off')

fig.suptitle('MNIST — Mẫu ảnh từng chữ số', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Phân phối pixel intensity ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram toàn bộ pixel
axes[0].hist(X_tr_mn.ravel(), bins=50, color='steelblue', alpha=0.8)
axes[0].set_title('Phân phối pixel (raw 0-255)')
axes[0].set_xlabel('Pixel value'); axes[0].set_ylabel('Tần số')

# Mean pixel theo từng feature (vị trí)
mean_img = X_tr_mn.mean(axis=0).reshape(H_mn, W_mn)
im = axes[1].imshow(mean_img, cmap='hot')
axes[1].set_title('Ảnh trung bình toàn tập train')
plt.colorbar(im, ax=axes[1])

plt.tight_layout(); plt.show()

In [ ]:
# ── PCA 2D Visualization (1000 mẫu) ───────────────────────────
N_VIZ = 1000
pca2 = PCA(n_components=2, random_state=42)
Z = pca2.fit_transform(X_tr_mn[:N_VIZ] / 255.0)   # normalize trước PCA

plt.figure(figsize=(8, 6))
for d in range(10):
    mask = y_tr_mn[:N_VIZ] == d
    plt.scatter(Z[mask, 0], Z[mask, 1], s=12, alpha=0.7, label=str(d))
plt.title(f'MNIST — PCA 2D ({N_VIZ} mẫu, var={pca2.explained_variance_ratio_.sum():.2%})')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='Chữ số', bbox_to_anchor=(1.05,1), loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

---
## 3. CIFAR-10 — Explore & Visualize

In [ ]:
X_tr_cf, y_tr_cf, X_te_cf, y_te_cf, cf_labels = load_cifar10_raw()

print("=" * 55)
print("CIFAR-10 — Thống kê cơ bản")
print("=" * 55)
print(f"X_train : {X_tr_cf.shape}  dtype={X_tr_cf.dtype}")
print(f"y_train : {y_tr_cf.shape}  classes={np.unique(y_tr_cf)}")
print(f"X_test  : {X_te_cf.shape}")
print(f"Pixel range  : [{X_tr_cf.min():.0f}, {X_tr_cf.max():.0f}]")
print(f"Mean (per channel):")
for ch, name in enumerate(['R','G','B']):
    c = X_tr_cf[:, ch*1024:(ch+1)*1024]
    print(f"  {name}: mean={c.mean():.2f}  std={c.std():.2f}")
print(f"Missing values: {np.isnan(X_tr_cf).sum()}")

In [ ]:
# ── Phân phối nhãn ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (y, split) in zip(axes, [(y_tr_cf,'Train'),(y_te_cf,'Test')]):
    counts = Counter(y)
    ax.bar([cf_labels[k] for k in sorted(counts)], 
           [counts[k] for k in sorted(counts)],
           color=sns.color_palette('tab10'))
    ax.set_title(f'CIFAR-10 — Phân phối nhãn ({split})')
    ax.set_xlabel('Class'); ax.set_ylabel('Số mẫu')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=35, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── Hiển thị ảnh mẫu (dạng RGB 32x32) ────────────────────────
def cifar_to_rgb(row):
    """Chuyển vector 3072 về ảnh 32x32x3."""
    return row.reshape(3, 32, 32).transpose(1, 2, 0).astype(np.uint8)

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for c in range(10):
    idxs = np.where(y_tr_cf == c)[0]
    for row, i in enumerate([idxs[0], idxs[1]]):
        axes[row, c].imshow(cifar_to_rgb(X_tr_cf[i]))
        if row == 0:
            axes[row, c].set_title(cf_labels[c], fontsize=8)
        axes[row, c].axis('off')

fig.suptitle('CIFAR-10 — Mẫu ảnh từng class', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Phân phối pixel theo 3 kênh RGB ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['red','green','blue']
for ch, (ax, color, name) in enumerate(zip(axes, colors, ['R','G','B'])):
    vals = X_tr_cf[:, ch*1024:(ch+1)*1024].ravel()
    ax.hist(vals, bins=50, color=color, alpha=0.7)
    ax.set_title(f'Kênh {name} — phân phối pixel')
    ax.set_xlabel('Pixel value'); ax.set_ylabel('Tần số')
plt.tight_layout(); plt.show()

In [ ]:
# ── Ảnh trung bình theo từng class ────────────────────────────
fig, axes = plt.subplots(1, 10, figsize=(16, 2.5))
for c in range(10):
    mask = y_tr_cf == c
    avg = X_tr_cf[mask].mean(axis=0)
    axes[c].imshow(cifar_to_rgb(avg))
    axes[c].set_title(cf_labels[c], fontsize=8)
    axes[c].axis('off')
fig.suptitle('CIFAR-10 — Ảnh trung bình từng class', fontsize=12)
plt.tight_layout(); plt.show()

---
## 4. RCV1 — Explore & Visualize

In [ ]:
X_tr_rc, y_tr_rc, X_te_rc, y_te_rc = load_rcv1_raw()

print("=" * 55)
print("RCV1 — Thống kê cơ bản")
print("=" * 55)
print(f"X_train : {X_tr_rc.shape}  type={type(X_tr_rc).__name__}")
print(f"y_train : {y_tr_rc.shape}  labels={np.unique(y_tr_rc)}")
print(f"X_test  : {X_te_rc.shape}")
print(f"Sparsity (train): {1 - X_tr_rc.nnz / (X_tr_rc.shape[0]*X_tr_rc.shape[1]):.4%}")
print(f"NNZ per sample  : {X_tr_rc.nnz / X_tr_rc.shape[0]:.1f}")

# Class balance
pos = (y_tr_rc == 1).sum()
neg = (y_tr_rc == -1).sum()
print(f"\nClass balance (train):")
print(f"  +1 (positive): {pos} ({pos/len(y_tr_rc):.2%})")
print(f"  -1 (negative): {neg} ({neg/len(y_tr_rc):.2%})")

In [ ]:
# ── Class balance & NNZ distribution ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Class balance
cnt = Counter(y_tr_rc)
axes[0].bar(['-1 (neg)', '+1 (pos)'], [cnt[-1.0], cnt[1.0]], 
            color=['#e74c3c','#2ecc71'])
axes[0].set_title('RCV1 — Phân phối nhãn (Train)')
axes[0].set_ylabel('Số mẫu')

# 2. NNZ per row (số feature khác 0 mỗi mẫu)
nnz_per_row = np.diff(X_tr_rc.indptr)
axes[1].hist(nnz_per_row, bins=60, color='steelblue', alpha=0.8)
axes[1].set_title('Số feature ≠ 0 mỗi mẫu')
axes[1].set_xlabel('NNZ'); axes[1].set_ylabel('Tần số')

# 3. Feature frequency (top-50 features dày nhất)
feat_freq = np.asarray(X_tr_rc.astype(bool).sum(axis=0)).ravel()
top_idx = np.argsort(feat_freq)[-50:]
axes[2].bar(range(50), feat_freq[top_idx], color='mediumpurple', alpha=0.8)
axes[2].set_title('Top 50 feature phổ biến nhất')
axes[2].set_xlabel('Rank'); axes[2].set_ylabel('Số tài liệu chứa feature')

plt.tight_layout(); plt.show()

---
## 5. Covtype — Explore & Visualize

In [ ]:
X_tr_cv, y_tr_cv, X_te_cv, y_te_cv = load_covtype_raw()

print("=" * 55)
print("Covtype — Thống kê cơ bản")
print("=" * 55)
print(f"X_train : {X_tr_cv.shape}  type={type(X_tr_cv).__name__}")
print(f"y_train : {y_tr_cv.shape}  labels={np.unique(y_tr_cv)}")
print(f"X_test  : {X_te_cv.shape}")

# Convert to dense để thống kê
X_cv_dense = X_tr_cv.toarray()
print(f"\nThống kê feature (dense):")
print(f"  Mean: {X_cv_dense.mean():.4f}")
print(f"  Std : {X_cv_dense.std():.4f}")
print(f"  Min : {X_cv_dense.min():.4f}")
print(f"  Max : {X_cv_dense.max():.4f}")

pos = (y_tr_cv == 1).sum()
neg = (y_tr_cv == -1).sum()
print(f"\nClass balance:")
print(f"  +1: {pos} ({pos/len(y_tr_cv):.2%})")
print(f"  -1: {neg} ({neg/len(y_tr_cv):.2%})")

In [ ]:
# ── Covtype Visualizations ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Class balance
cnt = Counter(y_tr_cv)
axes[0].bar(['-1', '+1'], [cnt[-1.0], cnt[1.0]], color=['#e74c3c','#2ecc71'])
axes[0].set_title('Covtype — Phân phối nhãn (Train)')
axes[0].set_ylabel('Số mẫu')

# 2. Feature means theo nhãn
X_pos = X_cv_dense[y_tr_cv == 1].mean(axis=0)
X_neg = X_cv_dense[y_tr_cv == -1].mean(axis=0)
x_idx = range(len(X_pos))
axes[1].plot(x_idx, X_pos, label='+1 (class 2)', alpha=0.8)
axes[1].plot(x_idx, X_neg, label='-1 (others)', alpha=0.8)
axes[1].set_title('Mean feature value theo nhãn')
axes[1].set_xlabel('Feature index'); axes[1].set_ylabel('Mean value')
axes[1].legend()

# 3. Feature std
feat_std = X_cv_dense.std(axis=0)
axes[2].bar(range(len(feat_std)), feat_std, color='coral', alpha=0.8)
axes[2].set_title('Độ lệch chuẩn từng feature')
axes[2].set_xlabel('Feature index'); axes[2].set_ylabel('Std')

plt.tight_layout(); plt.show()

In [ ]:
# ── Boxplot phân phối 10 feature đầu theo nhãn ─────────────────
import pandas as pd

n_feat_show = 10
df_cv = pd.DataFrame(X_cv_dense[:, :n_feat_show], 
                     columns=[f'f{i}' for i in range(n_feat_show)])
df_cv['label'] = y_tr_cv

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.ravel()):
    for label, color in [(-1.0,'#e74c3c'), (1.0,'#2ecc71')]:
        data = df_cv[df_cv['label'] == label][f'f{i}']
        ax.hist(data, bins=30, alpha=0.6, color=color, label=str(int(label)))
    ax.set_title(f'Feature {i}'); ax.set_xlabel('Value')
    if i == 0: ax.legend(title='Label')

fig.suptitle('Covtype — Phân phối 10 feature đầu theo nhãn', fontsize=13)
plt.tight_layout(); plt.show()

---
## 6. Tiền Xử Lý & Chuẩn Hóa

| Dataset  | Loại data  | Phương pháp chuẩn hóa           | Lý do                                     |
|----------|------------|----------------------------------|--------------------------------------------|
| MNIST    | Dense      | Min-Max [0,1] hoặc Standard Z    | Pixel đã đồng nhất, Z-score cho SVM/logistic|
| CIFAR-10 | Dense      | Per-channel mean/std (Z-score)   | RGB channels có phân phối khác nhau        |
| RCV1     | Sparse TF  | MaxAbsScaler (giữ sparsity)      | Sparse data, tránh mất zero entries        |
| Covtype  | Sparse mix | MaxAbsScaler (giữ sparsity)      | Mix continuous + binary features           |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  MNIST — 2 cách chuẩn hóa                                   ║
# ╚══════════════════════════════════════════════════════════════╝

# --- Cách 1: Min-Max [0, 1] (chia cho 255) ---
X_tr_mn_minmax = X_tr_mn / 255.0
X_te_mn_minmax = X_te_mn / 255.0

# --- Cách 2: StandardScaler (zero-mean, unit-variance) ---
sc_mn = StandardScaler()
X_tr_mn_std = sc_mn.fit_transform(X_tr_mn)
X_te_mn_std = sc_mn.transform(X_te_mn)

print("MNIST — Sau chuẩn hóa:")
print(f"  [Min-Max] range: [{X_tr_mn_minmax.min():.3f}, {X_tr_mn_minmax.max():.3f}]  mean={X_tr_mn_minmax.mean():.4f}")
print(f"  [Z-score] range: [{X_tr_mn_std.min():.3f}, {X_tr_mn_std.max():.3f}]  mean={X_tr_mn_std.mean():.6f}  std={X_tr_mn_std.std():.6f}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CIFAR-10 — Per-channel Z-score                             ║
# ╚══════════════════════════════════════════════════════════════╝

def cifar_per_channel_normalize(X):
    """Chuẩn hóa theo từng kênh R/G/B độc lập."""
    X_out = X.copy()
    for ch in range(3):
        s, e = ch * 1024, (ch + 1) * 1024
        mu  = X[:, s:e].mean()
        sig = X[:, s:e].std()
        X_out[:, s:e] = (X[:, s:e] - mu) / (sig + 1e-8)
    return X_out

# Tính mean/std trên TRAIN rồi áp vào TEST
ch_stats = []
for ch in range(3):
    s, e = ch * 1024, (ch + 1) * 1024
    mu  = X_tr_cf[:, s:e].mean()
    sig = X_tr_cf[:, s:e].std()
    ch_stats.append((mu, sig))

def cifar_normalize(X, ch_stats):
    X_out = X.copy()
    for ch, (mu, sig) in enumerate(ch_stats):
        s, e = ch * 1024, (ch + 1) * 1024
        X_out[:, s:e] = (X[:, s:e] - mu) / (sig + 1e-8)
    return X_out

X_tr_cf_norm = cifar_normalize(X_tr_cf, ch_stats)
X_te_cf_norm = cifar_normalize(X_te_cf, ch_stats)

print("CIFAR-10 — Sau per-channel Z-score:")
for ch, name in enumerate(['R','G','B']):
    s, e = ch * 1024, (ch + 1) * 1024
    c = X_tr_cf_norm[:, s:e]
    print(f"  {name}: mean={c.mean():.6f}  std={c.std():.6f}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  RCV1 — MaxAbsScaler (giữ sparsity)                        ║
# ╚══════════════════════════════════════════════════════════════╝

sc_rc = MaxAbsScaler()
X_tr_rc_norm = sc_rc.fit_transform(X_tr_rc)
X_te_rc_norm = sc_rc.transform(X_te_rc)

# Kiểm tra vẫn sparse
print("RCV1 — Sau MaxAbsScaler:")
print(f"  Type: {type(X_tr_rc_norm).__name__}  (sparse vẫn được giữ: {sp.issparse(X_tr_rc_norm)})")
print(f"  Max absolute value: {abs(X_tr_rc_norm).max():.6f}")
print(f"  NNZ unchanged: train={X_tr_rc_norm.nnz}, original={X_tr_rc.nnz}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Covtype — MaxAbsScaler (giữ sparsity)                     ║
# ╚══════════════════════════════════════════════════════════════╝

sc_cv = MaxAbsScaler()
X_tr_cv_norm = sc_cv.fit_transform(X_tr_cv)
X_te_cv_norm = sc_cv.transform(X_te_cv)

print("Covtype — Sau MaxAbsScaler:")
print(f"  Type: {type(X_tr_cv_norm).__name__}  sparse={sp.issparse(X_tr_cv_norm)}")
print(f"  Max absolute value: {abs(X_tr_cv_norm).max():.6f}")

---
## 7. So Sánh Trước / Sau Chuẩn Hóa

In [ ]:
# ── MNIST: before vs after ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(X_tr_mn.ravel(), bins=50, color='gray', alpha=0.8)
axes[0].set_title('MNIST — Raw (0-255)')

axes[1].hist(X_tr_mn_minmax.ravel(), bins=50, color='steelblue', alpha=0.8)
axes[1].set_title('MNIST — Min-Max [0, 1]')

axes[2].hist(X_tr_mn_std.ravel(), bins=50, color='tomato', alpha=0.8)
axes[2].set_title('MNIST — Z-score (mean≈0, std≈1)')

for ax in axes:
    ax.set_xlabel('Value'); ax.set_ylabel('Tần số')

plt.suptitle('MNIST — Pixel distribution: Raw vs Normalized', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── CIFAR-10: before vs after (kênh R) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(X_tr_cf[:, :1024].ravel(), bins=50, color='red', alpha=0.7)
axes[0].set_title('CIFAR-10 kênh R — Raw')

axes[1].hist(X_tr_cf_norm[:, :1024].ravel(), bins=50, color='darkred', alpha=0.7)
axes[1].set_title('CIFAR-10 kênh R — Sau Z-score')

for ax in axes:
    ax.set_xlabel('Value'); ax.set_ylabel('Tần số')
plt.tight_layout(); plt.show()

In [ ]:
# ── Covtype: before vs after (feature liên tục) ─────────────── 
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Feature 0 — Elevation (liên tục, range lớn)
axes[0].hist(X_cv_dense[:, 0], bins=50, color='teal', alpha=0.8)
axes[0].set_title('Covtype f0 (Elevation) — Raw')

axes[1].hist(X_tr_cv_norm[:, 0].toarray().ravel(), bins=50, color='darkcyan', alpha=0.8)
axes[1].set_title('Covtype f0 (Elevation) — Sau MaxAbsScaler')

for ax in axes:
    ax.set_xlabel('Value'); ax.set_ylabel('Tần số')
plt.tight_layout(); plt.show()

---
## 8. Tóm Tắt & Kiểm Tra Cuối

In [ ]:
# ── Bảng tổng kết ─────────────────────────────────────────────
import pandas as pd

summary = [
    {
        'Dataset': 'MNIST',
        'Train size': X_tr_mn_minmax.shape[0],
        'Test size' : X_te_mn_minmax.shape[0],
        'Features'  : X_tr_mn_minmax.shape[1],
        'Classes'   : 10,
        'Type'      : 'Dense',
        'Scaling'   : 'Min-Max [0,1] + Z-score option',
        'X_train range': f"[{X_tr_mn_minmax.min():.2f}, {X_tr_mn_minmax.max():.2f}]",
    },
    {
        'Dataset': 'CIFAR-10',
        'Train size': X_tr_cf_norm.shape[0],
        'Test size' : X_te_cf_norm.shape[0],
        'Features'  : X_tr_cf_norm.shape[1],
        'Classes'   : 10,
        'Type'      : 'Dense',
        'Scaling'   : 'Per-channel Z-score',
        'X_train range': f"[{X_tr_cf_norm.min():.2f}, {X_tr_cf_norm.max():.2f}]",
    },
    {
        'Dataset': 'RCV1',
        'Train size': X_tr_rc_norm.shape[0],
        'Test size' : X_te_rc_norm.shape[0],
        'Features'  : X_tr_rc_norm.shape[1],
        'Classes'   : 2,
        'Type'      : 'Sparse',
        'Scaling'   : 'MaxAbsScaler (giữ sparsity)',
        'X_train range': f"[{X_tr_rc_norm.min():.2f}, {X_tr_rc_norm.max():.2f}]",
    },
    {
        'Dataset': 'Covtype',
        'Train size': X_tr_cv_norm.shape[0],
        'Test size' : X_te_cv_norm.shape[0],
        'Features'  : X_tr_cv_norm.shape[1],
        'Classes'   : 2,
        'Type'      : 'Sparse',
        'Scaling'   : 'MaxAbsScaler (giữ sparsity)',
        'X_train range': f"[{X_tr_cv_norm.min():.2f}, {X_tr_cv_norm.max():.2f}]",
    },
]

df_summary = pd.DataFrame(summary).set_index('Dataset')
print("\n📋 TỔNG KẾT DATA SAU TIỀN XỬ LÝ")
print("=" * 70)
display(df_summary)

In [ ]:
# ── Kiểm tra không có NaN / Inf sau chuẩn hóa ─────────────────
checks = [
    ('MNIST   MinMax train', X_tr_mn_minmax),
    ('MNIST   Z-score train', X_tr_mn_std),
    ('CIFAR10 norm  train',  X_tr_cf_norm),
    ('RCV1    norm  train',  X_tr_rc_norm.toarray()),
    ('Covtype norm  train',  X_tr_cv_norm.toarray()),
]

print("\n🔍 Kiểm tra NaN / Inf sau chuẩn hóa:")
print(f"{'Dataset':<30} {'NaN':>6} {'Inf':>6} {'OK':>5}")
print("-" * 52)
all_ok = True
for name, X in checks:
    has_nan = np.isnan(X).sum()
    has_inf = np.isinf(X).sum()
    ok = (has_nan == 0 and has_inf == 0)
    if not ok: all_ok = False
    print(f"{name:<30} {has_nan:>6} {has_inf:>6} {'✅' if ok else '❌':>5}")

print()
if all_ok:
    print("✅ Tất cả datasets sạch — sẵn sàng đưa vào mô hình!")
else:
    print("❌ Có vấn đề — cần kiểm tra lại pipeline chuẩn hóa.")

---
## 9. Export Scalers (để dùng lại khi inference)

In [ ]:
import joblib, pathlib

SCALER_DIR = pathlib.Path(BASE) / 'scalers'
SCALER_DIR.mkdir(parents=True, exist_ok=True)

# Lưu MNIST StandardScaler (fit trên raw pixel)
joblib.dump(sc_mn, SCALER_DIR / 'mnist_standard_scaler.pkl')

# Lưu channel stats CIFAR-10 (dạng list of tuples)
joblib.dump(ch_stats, SCALER_DIR / 'cifar10_channel_stats.pkl')

# Lưu MaxAbsScaler cho RCV1 & Covtype
joblib.dump(sc_rc, SCALER_DIR / 'rcv1_maxabs_scaler.pkl')
joblib.dump(sc_cv, SCALER_DIR / 'covtype_maxabs_scaler.pkl')

print("✅ Scalers đã lưu vào:", SCALER_DIR)
for f in sorted(SCALER_DIR.glob('*.pkl')):
    print(f"   {f.name}")

---
## 10. Unified `load_dataset_normalized()` cho Model Training

In [ ]:
def load_dataset_normalized(name, mnist_mode='minmax'):
    """
    Load và chuẩn hóa dataset, trả về (X_train, y_train, X_test, y_test).
    
    Parameters
    ----------
    name : str — {'mnist', 'cifar10', 'rcv1', 'covtype'}
    mnist_mode : str — {'minmax', 'zscore'}  (chỉ dùng cho MNIST)
    
    Returns
    -------
    X_train, y_train, X_test, y_test  (numpy hoặc scipy sparse)
    """
    if name == 'mnist':
        X_tr, y_tr, X_te, y_te, H, W = load_mnist_raw()
        if mnist_mode == 'minmax':
            return X_tr / 255.0, y_tr, X_te / 255.0, y_te
        else:  # zscore
            sc = StandardScaler()
            return sc.fit_transform(X_tr), y_tr, sc.transform(X_te), y_te

    elif name == 'cifar10':
        X_tr, y_tr, X_te, y_te, _ = load_cifar10_raw()
        stats = []
        for ch in range(3):
            s, e = ch*1024, (ch+1)*1024
            stats.append((X_tr[:, s:e].mean(), X_tr[:, s:e].std()))
        return cifar_normalize(X_tr, stats), y_tr, cifar_normalize(X_te, stats), y_te

    elif name == 'rcv1':
        X_tr, y_tr, X_te, y_te = load_rcv1_raw()
        sc = MaxAbsScaler()
        return sc.fit_transform(X_tr), y_tr, sc.transform(X_te), y_te

    elif name == 'covtype':
        X_tr, y_tr, X_te, y_te = load_covtype_raw()
        sc = MaxAbsScaler()
        return sc.fit_transform(X_tr), y_tr, sc.transform(X_te), y_te

    else:
        raise ValueError(f"Unknown dataset: {name}")

# ── Kiểm tra nhanh ─────────────────────────────────────────────
for ds in ['mnist', 'cifar10', 'rcv1', 'covtype']:
    Xtr, ytr, Xte, yte = load_dataset_normalized(ds)
    print(f"{ds:10s}  train={Xtr.shape}  test={Xte.shape}  labels={np.unique(yte)}")